# 01b — Signal Scale Analysis: what timescale does ECG200 structure live at?

**Purpose.** The faithfulness harness (notebook 03) divides the 96-timestep signal into regions
whose size is set as a percentage of length — currently **2.5%** and **10%**, inherited from
Šimić, Veas & Sabol (2025). But that paper's signals were **much longer** than ECG200's 96
timesteps, so a percentage that was sensible there may not be here: 2.5% of a 500-point signal
is ~12 timesteps, while 2.5% of *our* 96 is only ~2–3. This notebook asks the question from the
**data itself**: over how many timesteps does ECG200's structure actually stay coherent, so we
can judge whether 2.5% / 10% resolve real features or slice them into fragments.

This is **analysis only** — it changes nothing in `src/xai/regions.py`. It produces evidence to
confirm, adjust, or add to the region sizes; we decide afterwards.

Two complementary estimates:
1. **Autocorrelation length** — a quantitative measure of the signal's natural correlation
   scale (how far apart two timesteps can be before they're effectively unrelated).
2. **Feature width** — a softer, domain-informed measure of how wide the discriminative
   features (the QRS-like deflection, the class-difference region) actually are.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_widths

from src.data.preprocessing import load_ecg200

# Use all samples for the autocorrelation estimate (more data = smoother curve);
# use the training set for per-class mean traces (matches notebook 01).
Xtr, ytr = load_ecg200("train")
Xva, yva = load_ecg200("val")
Xte, yte = load_ecg200("test")
X_all = np.vstack([Xtr, Xva, Xte])
y_all = np.concatenate([ytr, yva, yte])
N = X_all.shape[1]

FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("timesteps:", N, "| total samples:", X_all.shape[0])

## 1. Autocorrelation — the natural correlation length

**What autocorrelation is, in plain terms.** Take the signal and compare it against a copy of
itself shifted by *k* timesteps. If shifting by *k* still lines up well, timesteps that far
apart are still *related*; once the shifted copy stops matching, timesteps that far apart have
become effectively *independent*. Plotting this match-score against the shift *k* (the "lag")
gives the **autocorrelation function (ACF)**: it starts at 1.0 (a signal matches itself
perfectly at zero shift) and decays toward 0 as the lag grows.

**How to read it.** The lag at which the ACF has decayed to **1/e ≈ 0.37** is the conventional
**correlation length** — beyond that, neighbouring structure has largely washed out. The **first
zero crossing** marks where the signal becomes fully uncorrelated (and starts to anti-correlate).
That correlation length is our main quantitative estimate of the smallest chunk that still holds
a coherent piece of signal: a region much *smaller* than it is slicing inside one coherent feature.

In [ ]:
def mean_acf(X):
    acfs = []
    for x in X:
        x = x - x.mean()
        full = np.correlate(x, x, mode="full")
        acf = full[full.size // 2:]      # keep non-negative lags 0..N-1
        acfs.append(acf / acf[0])        # normalise so lag-0 == 1
    return np.mean(acfs, axis=0)

acf_all = mean_acf(X_all)
acf_c0  = mean_acf(X_all[y_all == 0])
acf_c1  = mean_acf(X_all[y_all == 1])
lags = np.arange(N)

def first_below(acf, thr):
    idx = np.where(acf < thr)[0]
    return int(idx[0]) if idx.size else None

thr_e = 1.0 / np.e
print(f"correlation length (first lag with ACF < 1/e = {thr_e:.3f}):")
print(f"  all classes : {first_below(acf_all, thr_e)} timesteps")
print(f"  class 0      : {first_below(acf_c0, thr_e)} timesteps")
print(f"  class 1      : {first_below(acf_c1, thr_e)} timesteps")
print(f"first zero crossing (all): {first_below(acf_all, 0.0)} timesteps")
print(f"ACF at lags 0..12 (all): {np.round(acf_all[:13], 3)}")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(lags, acf_all, color="k", lw=2, label="all samples")
ax.plot(lags, acf_c0, color="C3", lw=1, alpha=0.8, label="class 0 (abnormal)")
ax.plot(lags, acf_c1, color="C0", lw=1, alpha=0.8, label="class 1 (normal)")
ax.axhline(thr_e, color="grey", ls="--", lw=1, label="1/e ≈ 0.37")
ax.axhline(0.0, color="grey", lw=0.6)
cl = first_below(acf_all, thr_e)
ax.axvline(cl, color="C2", ls=":", lw=1.2, label=f"corr. length ≈ {cl} ts")
ax.set_xlim(0, 40); ax.set_xlabel("lag (timesteps)"); ax.set_ylabel("autocorrelation")
ax.set_title("ECG200 autocorrelation — natural correlation length")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "01b_autocorrelation.png", dpi=150, bbox_inches="tight")
plt.show()

### What the autocorrelation says

The ACF decays smoothly and reaches **1/e (≈0.37) at a lag of about 10 timesteps** — essentially
identical for both classes — and hits its **first zero crossing around lag 16**. In plain words:
**two points on an ECG200 beat stay meaningfully related for roughly 8–10 timesteps, and are
effectively independent by ~16.** So the signal's natural "grain" is on the order of **~10
timesteps**. Any region much smaller than that is carving up territory that is still internally
correlated — i.e. slicing within a single coherent piece of signal rather than isolating one.

## 2. Feature width — how wide are the discriminative features?

The autocorrelation is generic (it doesn't know about classes). As a second, domain-informed
check we ask how wide the features that actually *distinguish the two classes* are, using the
per-class mean traces from notebook 01. Two measures:

- **The dominant deflection width.** ECG200 beats are dominated by a large negative deflection
  (the QRS-like trough). We measure its **full width at half-prominence** — a standard way to
  say "how many timesteps wide is this feature."
- **Where the classes differ.** We take the absolute difference between the two class-mean
  traces and find the **smallest contiguous window that contains half of the total difference**,
  i.e. how localised (or spread) the class-discriminating signal is.

In [ ]:
m0 = Xtr[ytr == 0].mean(0)
m1 = Xtr[ytr == 1].mean(0)
t = np.arange(N)

# dominant trough width (half-prominence) per class
widths = {}
for name, m in [("class 0", m0), ("class 1", m1)]:
    inv = -m
    pk, props = find_peaks(inv, prominence=0.3)
    k = int(np.argmax(props["prominences"]))
    w = peak_widths(inv, [pk[k]], rel_height=0.5)[0][0]
    widths[name] = (int(pk[k]), w)
    print(f"{name}: dominant trough at t={pk[k]}, full width at half-prominence ≈ {w:.1f} ts")

# class-divergence concentration
div = np.abs(m1 - m0); divn = div / div.sum()
def smallest_window(frac):
    for w in range(1, N + 1):
        c = np.convolve(divn, np.ones(w), "valid")
        if c.max() >= frac:
            return w, int(c.argmax())
    return N, 0
w50, s50 = smallest_window(0.5)
print(f"class divergence: 50% of the class-difference mass fits in ~{w50} ts (starts ~t{s50})")
print(f"                  peak divergence at t={int(div.argmax())}")

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t, m0, color="C3", label="class 0 (abnormal)")
ax[0].plot(t, m1, color="C0", label="class 1 (normal)")
ax[0].set_ylabel("z-scored amplitude"); ax[0].legend(fontsize=8)
ax[0].set_title("Per-class mean traces")
ax[1].plot(t, div, color="C4")
ax[1].axvspan(s50, s50 + w50, color="C2", alpha=0.2, label=f"50% of divergence ({w50} ts)")
ax[1].set_ylabel("|class1 − class0|"); ax[1].set_xlabel("timestep"); ax[1].legend(fontsize=8)
ax[1].set_title("Where the classes differ")
fig.tight_layout()
fig.savefig(FIG_DIR / "01b_feature_width.png", dpi=150, bbox_inches="tight")
plt.show()

### What the feature widths say

The dominant QRS-like trough is **broad in the mean trace — roughly 24 timesteps wide (class 1)
to 37 (class 0)** at half-prominence — and the class-discriminating difference is **spread, not
spiky**: the smallest window holding half of the total class-divergence is **~26 timesteps**,
centred in the mid-beat region (peak difference near t≈43). Both measures agree that the
*meaningful features of ECG200 are wide* — tens of timesteps — and comfortably larger than the
~10-timestep autocorrelation length, which is what we'd expect (a feature contains several
correlation lengths). Nothing here lives at the 2–3 timestep scale.

## 3. Translating to region sizes

Putting the two estimates next to the current region choices (signal length = 96):

| Scale | Timesteps | As % of 96 |
|-------|-----------|------------|
| **Current fine grid (2.5%)** | 2.4 (regions of 2–3) | 2.5% |
| **Current coarse grid (10%)** | 9.6 (regions of 9–10) | 10% |
| **Autocorrelation length (1/e)** | **≈10** | ≈10% |
| **Autocorrelation zero crossing** | ≈16 | ≈17% |
| **Dominant feature width (FWHM)** | **≈24–37** | ≈25–38% |
| **Half of class-divergence mass** | **≈26** | ≈27% |

Reading this:

- **2.5% (2–3 ts) is well below the signal's natural scale.** It is ~4× finer than the
  correlation length and ~10× finer than the discriminative features. A 2–3 timestep region
  cannot contain a coherent feature — it is a *fragment* of one. Consequences downstream:
  perturbing a single such region removes only part of a feature (whose information is smeared
  over ~10 neighbouring, still-correlated timesteps), so single-region effects will be small and
  redundant, and per-region relevance will be noisier.
- **10% (9.6 ts) lands right on the correlation length (~10 ts).** This is the best-justified
  size in the data: a 10% region ≈ one natural "grain" of the signal — small enough to localise
  structure, large enough not to split a coherent unit. It is a genuinely sensible resolution.
- A size matched to the *whole* dominant feature would be even larger (~25%+), but that is
  probably too coarse for localising relevance; **~10% is the sweet spot** between resolving and
  fragmenting.

## 4. Recommendation

**Based on the ECG200 data, ~10% is the well-justified region size, and I recommend keeping it
as the primary grid.** It coincides with the signal's autocorrelation length (~10 timesteps), so
each region corresponds to roughly one coherent unit of signal. Conveniently this is *also* one
of Šimić's sizes, so here the data-justified and paper-comparable choices agree — no tradeoff.

**The 2.5% grid is the one to reconsider.** At 96 timesteps it collapses to 2–3-timestep
regions that sit well inside the signal's correlation length, so it fragments features rather
than resolving them. Two ways to handle it:

- **If data-fidelity is the priority:** replace the fine grid with **~5% (≈5 timesteps)** — still
  finer than 10% for a higher-resolution view, but no longer slicing far below the correlation
  length. This is the most defensible ECG200-native pair: **~5% and 10%**.
- **If literal comparability with Šimić is the priority:** keep **2.5% and 10%**, but document
  the caveat explicitly (below). 2.5% can still serve as a deliberately high-resolution relevance
  map — just don't over-interpret single fine regions.

**The tradeoff to decide, in plain terms.** Šimić's 2.5% was applied to *much longer* signals,
where 2.5% was many more absolute timesteps — plausibly near *their* correlation length. Because
ECG200 is only 96 points long, inheriting the *percentage* does **not** inherit the *effective
scale*: our 2.5% is far finer than theirs was, and in absolute terms **our 10% is closer in
spirit to their 2.5%**. So "keep 2.5%/10% for comparability" buys comparability of the *labels*,
not of the *physical region scale*. My suggestion: **keep 10% as the anchor (it satisfies both
criteria), and choose the fine grid deliberately — either 2.5% for numeric label-comparability
with the caveat noted, or ~5% for a better data-matched fine resolution.** No code changed; this
is for you to decide.